In [ ]:
import cv2
import numpy as np
import pandas as pd
import joblib
from google.colab.patches import cv2_imshow
from IPython.display import clear_output

In [ ]:
empty = []
numpy_array = np.array(empty)


In [ ]:
!pip install mediapipe opencv-python


In [ ]:
import mediapipe as mp


In [ ]:
# === Load models and scalers from Google Drive ===
model_dir = "/content/drive/MyDrive/MilesTone1_models/"

roll_model = joblib.load(model_dir + 'best_svr_roll_model.joblib')
pitch_model = joblib.load(model_dir + 'best_svr_pitch_model.joblib')
yaw_model = joblib.load(model_dir + 'best_svr_yaw_model.joblib')

roll_scaler = joblib.load(model_dir + 'scaler_roll.joblib')
pitch_scaler = joblib.load(model_dir + 'scaler_pitch.joblib')
yaw_scaler = joblib.load(model_dir + 'scaler_yaw.joblib')

# === MediaPipe config ===
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(min_detection_confidence=0.8, min_tracking_confidence=0.5)

# === Landmark indices ===
left_eye_landmarks = [33, 133, 160, 144, 153, 158]
right_eye_landmarks = [362, 263, 387, 373, 380, 385]
extra_landmarks = {
    137: 'landmark_137',
    366: 'landmark_366',
    1: 'nose_tip',
    152: 'chin_tip',
    61: 'mouth_left_corner',
    291: 'mouth_right_corner',
    10: 'forehead_center'
}

def calculate_eye_center(face_landmarks, indices):
    eye_coords = []
    for i in indices:
        x = face_landmarks.landmark[i].x
        y = face_landmarks.landmark[i].y
        z = face_landmarks.landmark[i].z
        eye_coords.append([x, y, z])
    return np.mean(eye_coords, axis=0)




# === Video path ===
video_path = "/content/drive/MyDrive/MilesTone1_models/WIN_20250429_00_04_49_Pro.mp4"
cap = cv2.VideoCapture(video_path)

# === Get video properties ===
fps = cap.get(cv2.CAP_PROP_FPS)
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# === Define codec and output path ===
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
output_path = 'final_processed_output.mp4'
out = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break


    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(frame_rgb)

    if results.multi_face_landmarks:
        face_landmarks = results.multi_face_landmarks[0]

        # Extract raw landmark values
        left_eye_center = calculate_eye_center(face_landmarks, left_eye_landmarks)
        right_eye_center = calculate_eye_center(face_landmarks, right_eye_landmarks)

        lm = face_landmarks.landmark
        landmark_vals = {}
        for idx, name in extra_landmarks.items():
            x = lm[idx].x
            y = lm[idx].y
            z = lm[idx].z
            landmark_vals[name] = (x, y, z)

        # Roll features
        roll_features = [
            landmark_vals['chin_tip'][0] - landmark_vals['forehead_center'][0],
            landmark_vals['chin_tip'][0] - landmark_vals['landmark_366'][0],
            landmark_vals['chin_tip'][0] - landmark_vals['landmark_137'][0],
        ]
        roll_input = pd.DataFrame([roll_features], columns=[
            'chin_minus_forehead_x', 'chin_minus_landmark366_x', 'chin_minus_landmark137_x'])
        roll_scaled = roll_scaler.transform(roll_input)
        roll_pred = roll_model.predict(roll_scaled)[0]

        # Pitch features
        pitch_features = [
            landmark_vals['nose_tip'][1] - landmark_vals['chin_tip'][1],
            landmark_vals['chin_tip'][2] - landmark_vals['forehead_center'][2],
            landmark_vals['nose_tip'][1] - landmark_vals['landmark_137'][1],
            landmark_vals['nose_tip'][1] - landmark_vals['landmark_366'][1],
        ]
        pitch_input = pd.DataFrame([pitch_features], columns=[
            'nose_minus_chin_y', 'chin_minus_forehead_z', 'nose_minus_landmark137_y', 'nose_minus_landmark366_y'])
        pitch_scaled = pitch_scaler.transform(pitch_input)
        pitch_pred = pitch_model.predict(pitch_scaled)[0]

        # Yaw features
        yaw_features = [
            landmark_vals['nose_tip'][0] - landmark_vals['mouth_left_corner'][0],
            landmark_vals['nose_tip'][0] - landmark_vals['mouth_right_corner'][0],
            landmark_vals['nose_tip'][0] - right_eye_center[0],
            landmark_vals['nose_tip'][0] - left_eye_center[0],
            landmark_vals['nose_tip'][0] - landmark_vals['landmark_137'][0],
            landmark_vals['nose_tip'][0] - landmark_vals['landmark_366'][0],
        ]
        yaw_input = pd.DataFrame([yaw_features], columns=[
            'nose_minus_mouth_left_x', 'nose_minus_mouth_right_x',
            'nose_minus_right_eye_x', 'nose_minus_left_eye_x',
            'nose_minus_landmark137_x', 'nose_minus_landmark366_x'])
        yaw_scaled = yaw_scaler.transform(yaw_input)
        yaw_pred = yaw_model.predict(yaw_scaled)[0]

        # Convert normalized coordinates to pixel for nose tip
        nose_x = int(landmark_vals['nose_tip'][0] * frame.shape[1])
        nose_y = int(landmark_vals['nose_tip'][1] * frame.shape[0])


        # Overlay predictions
        text = f"Roll: {roll_pred:.2f} | Pitch: {pitch_pred:.2f} | Yaw: {yaw_pred:.2f}"
        cv2.putText(frame, text, (30, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    # Write frame to output video
    out.write(frame)

    # Show frame in Colab
    clear_output(wait=True)
    cv2_imshow(frame)

# Release resources
cap.release()
out.release()
